In [ ]:
from __future__ import annotations

import re
import hashlib
from typing import Any
from urllib.parse import urljoin
from pathlib import Path

from playwright.async_api import async_playwright, Browser, BrowserContext, Page

ROOT_SELECTOR = "main, article, [role='main']"
OUT_DIR = Path("playwright_captures")
OUT_DIR.mkdir(exist_ok=True)


def clean_text(text: str | None) -> str:
    if not text:
        return ""
    return re.sub(r"\s+", " ", text).strip()


def normalize_href(href: str | None, base_url: str) -> str | None:
    if not href:
        return None
    href = href.strip()
    if not href or href.startswith("#") or href.startswith("javascript:"):
        return None
    abs_url = urljoin(base_url, href)
    if abs_url.startswith("mailto:") or abs_url.startswith("tel:"):
        return None
    return abs_url


def make_capture_name(url: str, suffix: str = "") -> str:
    digest = hashlib.md5(url.encode("utf-8")).hexdigest()[:10]
    return f"capture_{digest}{suffix}"


In [ ]:
async def auto_scroll(page: Page, step: int = 900, pause_ms: int = 300) -> None:
    await page.evaluate(
        """
        async ({ step, pauseMs }) => {
            await new Promise((resolve) => {
                let totalHeight = 0;
                let previousHeight = -1;

                const timer = setInterval(() => {
                    const scrollHeight = Math.max(
                        document.body.scrollHeight,
                        document.documentElement.scrollHeight
                    );

                    window.scrollBy(0, step);
                    totalHeight += step;

                    if (totalHeight >= scrollHeight || scrollHeight === previousHeight) {
                        clearInterval(timer);
                        window.scrollTo(0, 0);
                        resolve();
                    }

                    previousHeight = scrollHeight;
                }, pauseMs);
            });
        }
        """,
        {"step": step, "pauseMs": pause_ms},
    )
    await page.wait_for_timeout(1500)


async def launch_page(url: str) -> tuple[Any, Browser, BrowserContext, Page]:
    p = await async_playwright().start()
    browser = await p.chromium.launch(headless=True)
    context = await browser.new_context()
    page = await context.new_page()

    await page.goto(url, wait_until="domcontentloaded", timeout=45000)
    await page.wait_for_timeout(2000)
    await auto_scroll(page)
    await page.wait_for_timeout(1500)

    return p, browser, context, page


async def close_page(p: Any, browser: Browser, context: BrowserContext) -> None:
    await context.close()
    await browser.close()
    await p.stop()


def dedupe_links(links: list[dict[str, Any]], base_url: str) -> list[dict[str, Any]]:
    normalized_links: list[dict[str, Any]] = []
    seen: set[tuple[str, str]] = set()

    for link in links:
        href = normalize_href(link.get("href"), base_url)
        if not href:
            continue

        text = clean_text(link.get("text"))
        key = (href, text)
        if key in seen:
            continue
        seen.add(key)

        normalized_links.append(
            {
                **link,
                "href": href,
                "text": text,
                "title": clean_text(link.get("title")),
            }
        )

    return normalized_links


async def extract_frame_content(frame) -> dict[str, Any] | None:
    try:
        return await frame.evaluate(
            f"""
            () => {{
                const body = document.body;
                const main = document.querySelector("{ROOT_SELECTOR}");

                const clean = (s) => (s || "").replace(/\\s+/g, " ").trim();

                const isVisible = (el) => {{
                    const style = window.getComputedStyle(el);
                    const rect = el.getBoundingClientRect();
                    return style.display !== "none" &&
                           style.visibility !== "hidden" &&
                           rect.width > 0 &&
                           rect.height > 0;
                }};

                const abs = (href) => {{
                    try {{ return new URL(href, window.location.href).href; }}
                    catch {{ return null; }}
                }};

                const inlineText = (node) => {{
                    if (!node) return "";

                    if (node.nodeType === Node.TEXT_NODE) {{
                        return clean(node.textContent);
                    }}

                    if (node.nodeType !== Node.ELEMENT_NODE) {{
                        return "";
                    }}

                    const el = node;

                    if (!isVisible(el)) return "";

                    const tag = el.tagName.toLowerCase();

                    if (tag === "script" || tag === "style" || tag === "noscript") {{
                        return "";
                    }}

                    if (tag === "a") {{
                        const text = clean(el.innerText);
                        const href = abs(el.getAttribute("href"));
                        if (href && text) return `[${{text}}](${{href}})`;
                        if (href) return `[link](${{href}})`;
                        return text;
                    }}

                    const parts = [];
                    for (const child of el.childNodes) {{
                        const value = inlineText(child);
                        if (value) parts.push(value);
                    }}
                    return clean(parts.join(" "));
                }};

                const blockText = (root) => {{
                    if (!root) return "";

                    const selectors = "h1,h2,h3,h4,h5,h6,article,section,li,p,div,tr,td";
                    const blocks = [...root.querySelectorAll(selectors)]
                        .filter(isVisible)
                        .map(el => {{
                            const tag = el.tagName.toLowerCase();
                            const text = inlineText(el);
                            if (!text) return null;

                            if (tag.match(/^h[1-6]$/)) {{
                                return `<${{tag}}>` + text + `</${{tag}}>`;
                            }}

                            if (tag === "li") {{
                                return `<li>` + text + `</li>`;
                            }}

                            if (tag === "article" || tag === "section") {{
                                return `\\n<${{tag}}>\\n` + text + `\\n</${{tag}}>\\n`;
                            }}

                            return `<${{tag}}>` + text + `</${{tag}}>`;
                        }})
                        .filter(Boolean);

                    return clean(blocks.join("\\n\\n"));
                }};

                const extractHeadings = (root) =>
                    [...root.querySelectorAll("h1,h2,h3,h4,h5,h6")]
                        .filter(isVisible)
                        .map(el => ({{
                            level: Number(el.tagName.slice(1)),
                            text: clean(el.innerText)
                        }}))
                        .filter(x => x.text);

                const extractLinks = (root) =>
                    [...root.querySelectorAll("a[href]")]
                        .filter(isVisible)
                        .map(a => ({{
                            text: clean(a.innerText),
                            href: a.getAttribute("href"),
                            title: clean(a.getAttribute("title"))
                        }}))
                        .filter(x => x.href);

                if (!body) return null;

                return {{
                    frame_url: window.location.href,
                    body_text: blockText(body),
                    main_text: main ? blockText(main) : "",
                    headings: extractHeadings(body),
                    links: extractLinks(body)
                }};
            }}
            """
        )
    except Exception:
        return None

async def capture_page(url: str) -> dict[str, Any]:
    p, browser, context, page = await launch_page(url)
    try:
        file_base = make_capture_name(url)
        screenshot_path = OUT_DIR / f"{file_base}.png"

        await page.screenshot(path=str(screenshot_path), full_page=True)

        data = await page.evaluate(
            f"""
            () => {{
                const body = document.body;
                const main = document.querySelector("{ROOT_SELECTOR}");
                const MAX_INLINE_LINK_LENGTH = 600;

                const clean = (s) => (s || "").replace(/\\s+/g, " ").trim();

                const isVisible = (el) => {{
                    const style = window.getComputedStyle(el);
                    const rect = el.getBoundingClientRect();
                    return style.display !== "none" &&
                           style.visibility !== "hidden" &&
                           rect.width > 0 &&
                           rect.height > 0;
                }};

                const abs = (href) => {{
                    try {{ return new URL(href, window.location.href).href; }}
                    catch {{ return null; }}
                }};

                const inlineText = (node) => {{
                    if (!node) return "";

                    if (node.nodeType === Node.TEXT_NODE) {{
                        return clean(node.textContent);
                    }}

                    if (node.nodeType !== Node.ELEMENT_NODE) {{
                        return "";
                    }}

                    const el = node;

                    if (!isVisible(el)) return "";

                    const tag = el.tagName.toLowerCase();

                    if (tag === "script" || tag === "style" || tag === "noscript") {{
                        return "";
                    }}

                    if (tag === "a") {{
                        const text = clean(el.innerText);
                        const href = abs(el.getAttribute("href"));

                        if (href && href.length <= MAX_INLINE_LINK_LENGTH && text) {{
                            return `[${{text}}](${{href}})`;
                        }}

                        if (text) return text;
                        return "";
                    }}

                    const parts = [];
                    for (const child of el.childNodes) {{
                        const value = inlineText(child);
                        if (value) parts.push(value);
                    }}
                    return clean(parts.join(" "));
                }};

                const extractTextBlocks = (root) => {{
                    const selectors = "h1,h2,h3,h4,h5,h6,article,section,li,p,div,tr,td";
                    const rawBlocks = [...root.querySelectorAll(selectors)]
                        .filter(isVisible)
                        .map(el => {{
                            const tag = el.tagName.toLowerCase();
                            const text = inlineText(el);
                            if (!text || text.length <= 20) return null;
                            return {{
                                tag,
                                text,
                                html_marker: `<${{tag}}>...</${{tag}}>`,
                            }};
                        }})
                        .filter(Boolean);

                    const deduped = [];
                    const seenRecent = [];

                    for (const block of rawBlocks) {{
                        const key = block.text;
                        if (seenRecent.includes(key)) continue;

                        deduped.push(block);
                        seenRecent.push(key);
                        if (seenRecent.length > 8) seenRecent.shift();
                    }}

                    return deduped.slice(0, 1000);
                }};

                const blockText = (root) => {{
                    if (!root) return "";
                    return extractTextBlocks(root)
                        .map(b => b.html_marker.replace("...", b.text))
                        .join("\\n\\n");
                }};

                const extractHeadings = (root) =>
                    [...root.querySelectorAll("h1,h2,h3,h4,h5,h6")]
                        .filter(isVisible)
                        .map(el => ({{
                            level: Number(el.tagName.slice(1)),
                            text: clean(el.innerText)
                        }}))
                        .filter(x => x.text);

                const extractLinks = (root) =>
                    [...root.querySelectorAll("a[href]")]
                        .filter(isVisible)
                        .map(a => ({{
                            text: clean(a.innerText),
                            href: a.getAttribute("href"),
                            title: clean(a.getAttribute("title"))
                        }}))
                        .filter(x => x.href);

                return {{
                    title: document.title,
                    final_url: window.location.href,
                    body_text: blockText(body),
                    main_text: main ? blockText(main) : "",
                    headings: extractHeadings(body || document.documentElement),
                    main_headings: main ? extractHeadings(main) : [],
                    links: extractLinks(body || document.documentElement),
                    main_links: main ? extractLinks(main) : [],
                    body_html: body ? body.innerHTML : "",
                    main_html: main ? main.innerHTML : ""
                }};
            }}
            """
        )

        data["links"] = dedupe_links(data["links"], data["final_url"])
        data["main_links"] = dedupe_links(data["main_links"], data["final_url"])
        data["screenshot_path"] = str(screenshot_path)

        frame_summaries: list[dict[str, Any]] = []
        for frame in page.frames:
            if frame == page.main_frame:
                continue

            frame_data = await extract_frame_content(frame)
            if not frame_data:
                continue

            frame_links = dedupe_links(
                frame_data.get("links", []),
                frame_data.get("frame_url", data["final_url"])
            )

            if not clean_text(frame_data.get("body_text")) and not frame_links:
                continue

            frame_summaries.append(
                {
                    "frame_url": frame_data.get("frame_url"),
                    "body_text": frame_data.get("body_text", ""),
                    "main_text": frame_data.get("main_text", ""),
                    "headings": frame_data.get("headings", []),
                    "links": frame_links,
                    "diagnostics": {
                        "body_text_length": len(frame_data.get("body_text", "")),
                        "main_text_length": len(frame_data.get("main_text", "")),
                        "link_count": len(frame_links),
                    },
                }
            )

        return {
            "url": url,
            **data,
            "frame_summaries": frame_summaries,
            "diagnostics": {
                "body_text_length": len(data["body_text"]),
                "main_text_length": len(data["main_text"]),
                "link_count": len(data["links"]),
                "main_link_count": len(data["main_links"]),
                "frame_count": len(page.frames),
                "captured_frame_count": len(frame_summaries),
            },
        }
    finally:
        await close_page(p, browser, context)


async def capture_page_chunks(url: str) -> dict[str, Any]:
    p, browser, context, page = await launch_page(url)
    try:
        file_base = make_capture_name(url, "_chunks")
        screenshot_path = OUT_DIR / f"{file_base}.png"

        await page.screenshot(path=str(screenshot_path), full_page=True)

        data = await page.evaluate(
            f"""
            () => {{
                const body = document.body;
                const main = document.querySelector("{ROOT_SELECTOR}");
                const MAX_INLINE_LINK_LENGTH = 600;

                const clean = (s) => (s || "").replace(/\\s+/g, " ").trim();

                const isVisible = (el) => {{
                    const style = window.getComputedStyle(el);
                    const rect = el.getBoundingClientRect();
                    return style.display !== "none" &&
                           style.visibility !== "hidden" &&
                           rect.width > 0 &&
                           rect.height > 0;
                }};

                const abs = (href) => {{
                    try {{ return new URL(href, window.location.href).href; }}
                    catch {{ return null; }}
                }};

                const inlineText = (node) => {{
                    if (!node) return "";

                    if (node.nodeType === Node.TEXT_NODE) {{
                        return clean(node.textContent);
                    }}

                    if (node.nodeType !== Node.ELEMENT_NODE) {{
                        return "";
                    }}

                    const el = node;

                    if (!isVisible(el)) return "";

                    const tag = el.tagName.toLowerCase();

                    if (tag === "script" || tag === "style" || tag === "noscript") {{
                        return "";
                    }}

                    if (tag === "a") {{
                        const text = clean(el.innerText);
                        const href = abs(el.getAttribute("href"));

                        if (href && href.length <= MAX_INLINE_LINK_LENGTH && text) {{
                            return `[${{text}}](${{href}})`;
                        }}

                        if (text) return text;
                        return "";
                    }}

                    const parts = [];
                    for (const child of el.childNodes) {{
                        const value = inlineText(child);
                        if (value) parts.push(value);
                    }}
                    return clean(parts.join(" "));
                }};

                const extractHeadings = (root) =>
                    [...root.querySelectorAll("h1,h2,h3,h4,h5,h6")]
                        .filter(isVisible)
                        .map(el => ({{
                            level: Number(el.tagName.slice(1)),
                            text: clean(el.innerText)
                        }}))
                        .filter(x => x.text);

                const extractLinks = (root) =>
                    [...root.querySelectorAll("a[href]")]
                        .filter(isVisible)
                        .map(a => ({{
                            text: clean(a.innerText),
                            href: a.getAttribute("href"),
                            title: clean(a.getAttribute("title"))
                        }}))
                        .filter(x => x.href);

                const extractTextBlocks = (root) => {{
                    const selectors = "h1,h2,h3,h4,h5,h6,article,section,li,p,div,tr,td";
                    const rawBlocks = [...root.querySelectorAll(selectors)]
                        .filter(isVisible)
                        .map(el => {{
                            const tag = el.tagName.toLowerCase();
                            const text = inlineText(el);
                            if (!text || text.length <= 20) return null;
                            return {{
                                tag,
                                text,
                                html_marker: `<${{tag}}>...</${{tag}}>`,
                            }};
                        }})
                        .filter(Boolean);

                    const deduped = [];
                    const seenRecent = [];

                    for (const block of rawBlocks) {{
                        const key = block.text;
                        if (seenRecent.includes(key)) continue;

                        deduped.push(block);
                        seenRecent.push(key);
                        if (seenRecent.length > 8) seenRecent.shift();
                    }}

                    return deduped.slice(0, 1000);
                }};

                const blockText = (root) => {{
                    if (!root) return "";
                    return extractTextBlocks(root)
                        .map(b => b.html_marker.replace("...", b.text))
                        .join("\\n\\n");
                }};

                return {{
                    title: document.title,
                    final_url: window.location.href,
                    body_text: blockText(body),
                    main_text: main ? blockText(main) : "",
                    headings: extractHeadings(body || document.documentElement),
                    main_headings: main ? extractHeadings(main) : [],
                    body_text_blocks: extractTextBlocks(body || document.documentElement),
                    main_text_blocks: main ? extractTextBlocks(main) : [],
                    links: extractLinks(body || document.documentElement),
                    main_links: main ? extractLinks(main) : [],
                    body_html: body ? body.innerHTML : "",
                    main_html: main ? main.innerHTML : ""
                }};
            }}
            """
        )

        data["links"] = dedupe_links(data["links"], data["final_url"])
        data["main_links"] = dedupe_links(data["main_links"], data["final_url"])
        data["screenshot_path"] = str(screenshot_path)

        frame_summaries: list[dict[str, Any]] = []
        for frame in page.frames:
            if frame == page.main_frame:
                continue

            frame_data = await extract_frame_content(frame)
            if not frame_data:
                continue

            frame_links = dedupe_links(
                frame_data.get("links", []),
                frame_data.get("frame_url", data["final_url"])
            )

            if not clean_text(frame_data.get("body_text")) and not frame_links:
                continue

            frame_summaries.append(
                {
                    "frame_url": frame_data.get("frame_url"),
                    "body_text": frame_data.get("body_text", ""),
                    "main_text": frame_data.get("main_text", ""),
                    "headings": frame_data.get("headings", []),
                    "links": frame_links,
                    "diagnostics": {
                        "body_text_length": len(frame_data.get("body_text", "")),
                        "main_text_length": len(frame_data.get("main_text", "")),
                        "link_count": len(frame_links),
                    },
                }
            )

        return {
            "url": url,
            **data,
            "frame_summaries": frame_summaries,
            "diagnostics": {
                "body_text_length": len(data["body_text"]),
                "main_text_length": len(data["main_text"]),
                "body_text_block_count": len(data["body_text_blocks"]),
                "main_text_block_count": len(data["main_text_blocks"]),
                "link_count": len(data["links"]),
                "main_link_count": len(data["main_links"]),
                "frame_count": len(page.frames),
                "captured_frame_count": len(frame_summaries),
            },
        }
    finally:
        await close_page(p, browser, context)

In [ ]:
listing = await capture_page("https://www.jacksonholechamber.com/events/")
print(len(listing["links"]))

In [ ]:
print(listing["body_text"])

In [ ]:
detail = await capture_page_chunks("https://www.jacksonholechamber.com/event/campfire-community-discussion/3168/")
print(detail["diagnostics"])
print(detail["screenshot_path"])
print(detail["body_text_blocks"][:5])